In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- medmodels_classic_distance_covariates_select_migration ---
FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET = pd.DataFrame({"age":[45,60,35],"bmi":[25.0,28.5,22.1],"risk":[0.2,0.8,0.1]})
FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET = pd.DataFrame({"age":[46,58],"bmi":[24.5,29.0],"risk":[0.3,0.7]})
FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET_PL = pl.from_pandas(FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET)
FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET_PL = pl.from_pandas(FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET)

# --- medmodels_classic_distance_match_migration ---
def _euclidean(a, b):
    return float(np.sqrt(np.sum((a - b) ** 2)))

def _mahalanobis(a, b, inv_cov=None):
    diff = a - b
    return float(np.sqrt(diff @ inv_cov @ diff.T)) if np.ndim(inv_cov) == 2 else float(abs(diff).sum())

FIX_MEDMODELS_CLASSIC_DISTANCE_MATCH_MIGRATION_METRICS = SimpleNamespace(
    METRICS={"euclidean": _euclidean},
    mahalanobis_metric=_mahalanobis,
)

try:
    pd.Index.__class_getitem__ = classmethod(lambda cls, item: cls)
except Exception:
    pass

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_medmodels_classic_distance_covariates_select_migration(control_set, treated_set, covariates=None):
    columns = treated_set.columns

    if not covariates:
        covariates = columns

    treated_array = treated_set[covariates].to_numpy().astype(float)
    control_array = control_set[covariates].to_numpy().astype(float)
    return control_array

def before_medmodels_classic_distance_match_migration(metrics):
    def nearest_neighbor(
        treated_set: pd.DataFrame,
        control_set: pd.DataFrame,
        metric: str,
        covariates: Optional[Union[List[str], pd.Index[str]]] = None,
    ) -> pd.DataFrame:
        columns = treated_set.columns

        if not covariates:
            covariates = columns

        treated_array = treated_set[covariates].to_numpy().astype(float)
        control_array = control_set[covariates].to_numpy().astype(float)
        control_array_full = control_set.to_numpy()
        matched_group = pd.DataFrame(columns=columns)

        cov = np.array([])
        if metric == "mahalanobis":
            cov = np.cov(np.concatenate((treated_array, control_array)).T)

        for element_ss in treated_array:
            if metric == "mahalanobis":
                if cov.ndim == 0:
                    inv_cov = 1 / cov
                else:
                    try:
                        inv_cov = np.linalg.inv(cov)
                    except np.linalg.LinAlgError:
                        raise ValueError(
                            "The covariance matrix is singular. Please, check the data."
                        )

                dist = [
                    metrics.mahalanobis_metric(element_ss, element_bs, inv_cov=inv_cov)
                    for element_bs in control_array
                ]

            else:
                metric_function = metrics.METRICS[metric]

                dist = [
                    metric_function(element_ss, element_bs) for element_bs in control_array
                ]

            nn_index = np.argmin(dist)

            new_row = pd.DataFrame(control_array_full[nn_index], index=columns)
            matched_group = (
                new_row.transpose().astype(float).copy()
                if matched_group.empty
                else pd.concat([matched_group, new_row.transpose().astype(float)])
            )
            control_array_full = np.delete(control_array_full, nn_index, 0)
            control_array = np.delete(control_array, nn_index, 0)

        return matched_group.reset_index(drop=True)
    return nearest_neighbor

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_medmodels_classic_distance_covariates_select_migration(control_set, treated_set, covariates=None):
    columns = treated_set.columns

    if not covariates:
        covariates = columns

    treated_array = treated_set.select(covariates).to_numpy().astype(float)
    control_array = control_set.select(covariates).to_numpy().astype(float)
    return control_array

def gen_medmodels_classic_distance_match_migration(metrics):
    def nearest_neighbor(
        treated_set: pl.DataFrame,
        control_set: pl.DataFrame,
        metric: str,
        covariates: Optional[Union[List[str], pl.Series]] = None,
    ) -> pl.DataFrame:
        columns = treated_set.columns

        if not covariates:
            covariates = columns

        treated_array = treated_set.select(covariates).to_numpy().astype(float)
        control_array = control_set.select(covariates).to_numpy().astype(float)
        control_array_full = control_set.to_numpy()
        matched_group = pl.DataFrame(schema={col: pl.Float64 for col in columns})

        cov = np.array([])
        if metric == "mahalanobis":
            cov = np.cov(np.concatenate((treated_array, control_array)).T)

        for element_ss in treated_array:
            if metric == "mahalanobis":
                if cov.ndim == 0:
                    inv_cov = 1 / cov
                else:
                    try:
                        inv_cov = np.linalg.inv(cov)
                    except np.linalg.LinAlgError:
                        raise ValueError(
                            "The covariance matrix is singular. Please, check the data."
                        )

                dist = [
                    metrics.mahalanobis_metric(element_ss, element_bs, inv_cov=inv_cov)
                    for element_bs in control_array
                ]

            else:
                metric_function = metrics.METRICS[metric]

                dist = [
                    metric_function(element_ss, element_bs) for element_bs in control_array
                ]

            nn_index = np.argmin(dist)

            new_row = pl.DataFrame([control_array_full[nn_index].tolist()], schema=columns)
            matched_group = (
                new_row.cast(pl.Float64)
                if matched_group.is_empty()
                else pl.concat([matched_group, new_row.cast(pl.Float64)], how="vertical")
            )
            control_array_full = np.delete(control_array_full, nn_index, 0)
            control_array = np.delete(control_array, nn_index, 0)

        return matched_group
    return nearest_neighbor

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: medmodels_classic_distance_covariates_select_migration ===

# L1 smoke – generated
try:
    _r = gen_medmodels_classic_distance_covariates_select_migration(FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET_PL, FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET_PL)
    print("✅ L1 smoke gen_medmodels_classic_distance_covariates_select_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_medmodels_classic_distance_covariates_select_migration: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_medmodels_classic_distance_covariates_select_migration(FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET, FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET)
    print("✅ L1 smoke before_medmodels_classic_distance_covariates_select_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_medmodels_classic_distance_covariates_select_migration: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – numpy array output.
try:
    _rb = before_medmodels_classic_distance_covariates_select_migration(FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET, FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET)
    _rg = gen_medmodels_classic_distance_covariates_select_migration(FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET_PL, FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET_PL)
    if np.allclose(_rb, _rg):
        print("✅ L2 equivalence medmodels_classic_distance_covariates_select_migration: MATCH")
    else:
        print(f"❌ L2 equivalence medmodels_classic_distance_covariates_select_migration: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence medmodels_classic_distance_covariates_select_migration: setup error — {type(_e).__name__}: {_e}")

# L3 branch – explicit covariates
try:
    _before_edge = before_medmodels_classic_distance_covariates_select_migration(
        FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET,
        FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET,
        covariates=["age", "bmi"],
    )
    _gen_edge = gen_medmodels_classic_distance_covariates_select_migration(
        FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_CONTROL_SET_PL,
        FIX_MEDMODELS_CLASSIC_DISTANCE_COVARIATES_SELECT_MIGRATION_TREATED_SET_PL,
        covariates=["age", "bmi"],
    )
    if np.allclose(_before_edge, _gen_edge):
        print("✅ L3 branch medmodels_classic_distance_covariates_select_migration: MATCH")
    else:
        print(f"❌ L3 branch medmodels_classic_distance_covariates_select_migration: MISMATCH — before={_before_edge}, gen={_gen_edge}")
except Exception as _e:
    print(f"❌ L3 branch medmodels_classic_distance_covariates_select_migration: {type(_e).__name__}: {_e}")
